<a href="https://colab.research.google.com/github/slavyolov/Algorithms/blob/main/RAG%2C_Llama%2C_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install llama-index llama-index-llms-groq groq -qq

To use Groq, you'll need an API key. If you don't already have one, create one in the Groq console. In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GROQ_API_KEY`. Then, we can securely access it.

In [12]:
# Used to securely store your API key
from google.colab import userdata

import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

Now, let's initialize the Groq LLM with `llama-index`.

In [13]:
from llama_index.llms.groq import Groq

# https://www.grizzlypeaksoftware.com/articles/p/groq-api-free-tier-limits-in-2026-what-you-actually-get-uwysd6mb
MODEL_NAME = "llama-3.1-8b-instant"
llm = Groq(model=MODEL_NAME)

# Step 2: Verify that the model works

System prompt for the LLM (needed for validation).

Now, let's make a query using our initialized Groq LLM with a system prompt.

In [14]:
response = llm.complete("What are the recent advancements in Large Language Models?")
print(response.text)

Recent advancements in Large Language Models (LLMs) have been significant, with several breakthroughs in the field. Here are some of the key developments:

1. **Transformers and Attention Mechanisms**: The Transformer architecture, introduced in 2017, has become the de facto standard for LLMs. The attention mechanism, which allows the model to focus on specific parts of the input, has been instrumental in achieving state-of-the-art results.
2. **Scaling Up**: The size of LLMs has increased dramatically, with models like BERT (12-layer, 340M parameters), RoBERTa (24-layer, 355M parameters), and T5 (24-layer, 770M parameters) pushing the boundaries of what is possible.
3. **Pre-training and Fine-tuning**: Pre-training on large datasets and fine-tuning on specific tasks has become a standard approach. This allows LLMs to learn general language understanding and then adapt to specific tasks.
4. **Multitask Learning**: LLMs are now being trained on multiple tasks simultaneously, such as lan

# Step 3- Configuring embeddings and LlamaIndex defaults

In [15]:
!pip install llama-index-embeddings-huggingface -qq

In [16]:
from llama_index.core import Settings
from llama_index.core.embeddings import resolve_embed_model

embed_model = resolve_embed_model("local:BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

# Step 4- Generating two demo PDFs

In [17]:
# Install ReportLab (needed to create pdf files)
!pip install -q reportlab

In [18]:
import os
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

# Ensure subdirectory exists
subdir = "demo_pdfs"
os.makedirs(subdir, exist_ok=True)

def create_pdf(path, title, paragraphs):
    """
    Create a simple multi-page PDF with a title and a list of text paragraphs.
    """
    c = canvas.Canvas(path, pagesize=A4)
    width, height = A4

    # Title
    c.setFont("Helvetica-Bold", 18)
    c.drawString(72, height - 72, title)

    # Body text
    c.setFont("Helvetica", 11)
    y = height - 110

    for p in paragraphs:
        for line in p.split("\n"):
            c.drawString(72, y, line)
            y -= 16

            # If we're near the bottom, start a new page
            if y < 72:
                c.showPage()
                c.setFont("Helvetica", 11)
                y = height - 72

    c.showPage()
    c.save()

pdf1_path = os.path.join(subdir, "project_overview.pdf")
pdf2_path = os.path.join(subdir, "agentic_ai_notes.pdf")

create_pdf(
    pdf1_path,
    "Project Overview: RAG Demo",
    [
        "This PDF describes example projects that combine time series forecasting, "
        "optimization, and agentic AI.",
        "It is generated programmatically as part of a demo to show RAG over user-created PDFs.",
        "Key topics include: demand forecasting, walk-forward validation, "
        "multi-agent research workflows, and LLM-powered 'talk to your data' tools.",
    ],
)

create_pdf(
    pdf2_path,
    "Agentic AI Notes",
    [
        "These notes outline concepts like multi-agent systems, tool-calling, and agent workflows.",
        "They are used in a LlamaIndex demo to illustrate how local LLMs can answer "
        "questions over private documentation.",
        "We combine embeddings, a vector index, and a simple agent to query these PDFs.",
    ],
)

print("Created PDFs:")
print(pdf1_path)
print(pdf2_path)

Created PDFs:
demo_pdfs/project_overview.pdf
demo_pdfs/agentic_ai_notes.pdf


# Step 4 - Building the RAG index over the PDFs

In [19]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load all PDFs from the demo_pdfs directory
docs = SimpleDirectoryReader(input_dir="demo_pdfs").load_data()
print(f"Loaded {len(docs)} documents")
for d in docs:
    print(d.metadata)

# Build an in-memory vector index over the documents
index = VectorStoreIndex.from_documents(docs)

# Expose a simple query engine (RAG interface)
query_engine = index.as_query_engine(similarity_top_k=3)

Loaded 2 documents
{'page_label': '1', 'file_name': 'agentic_ai_notes.pdf', 'file_path': '/content/demo_pdfs/agentic_ai_notes.pdf', 'file_type': 'application/pdf', 'file_size': 1782, 'creation_date': '2026-06-04', 'last_modified_date': '2026-06-04'}
{'page_label': '1', 'file_name': 'project_overview.pdf', 'file_path': '/content/demo_pdfs/project_overview.pdf', 'file_type': 'application/pdf', 'file_size': 1830, 'creation_date': '2026-06-04', 'last_modified_date': '2026-06-04'}


In [20]:
response = query_engine.query("What kinds of projects are described in these documents?")
print(response)

The projects described in these documents combine time series forecasting, optimization, and agentic AI. They include example projects that showcase various techniques and tools.


In [21]:
chat_engine = index.as_chat_engine(similarity_top_k=3)

response = chat_engine.chat(
    "Summarize the key ideas about agentic AI and RAG mentioned in these PDFs."
)
print(response)

Based on the provided documents, I'd be happy to summarize the key ideas about agentic AI and RAG.

**Agentic AI:**

From the "Agentic AI Notes" document, we can see that agentic AI is related to concepts like:

1. **Multi-agent systems**: This refers to a system composed of multiple interacting agents, which can be software or hardware components that work together to achieve a common goal.
2. **Tool-calling**: This is a mechanism that allows agents to call upon other tools or services to perform specific tasks or functions.
3. **Agent workflows**: This refers to the sequence of actions or tasks that an agent performs to achieve a specific goal or objective.

In the context of the LlamaIndex demo, agentic AI is used to illustrate how local LLMs (Large Language Models) can answer questions over private documentation.

**RAG (Reformer-based Aggregator):**

From the "Project Overview: RAG Demo" document, we can see that RAG is a key component of the demo, which combines time series forec

In [22]:
# Inspect what was actually retrieved for the last query
for i, sn in enumerate(response.source_nodes):
    print(f"\n--- Source node {i} (score={sn.score:.3f}) ---")
    print("Metadata:", sn.node.metadata)
    print(sn.node.get_content()[:400])


--- Source node 0 (score=0.806) ---
Metadata: {'page_label': '1', 'file_name': 'agentic_ai_notes.pdf', 'file_path': '/content/demo_pdfs/agentic_ai_notes.pdf', 'file_type': 'application/pdf', 'file_size': 1782, 'creation_date': '2026-06-04', 'last_modified_date': '2026-06-04'}
Agentic AI Notes
These notes outline concepts like multi-agent systems, tool-calling, and agent workflows.
They are used in a LlamaIndex demo to illustrate how local LLMs can answer questions over private documentation.
We combine embeddings, a vector index, and a simple agent to query these PDFs.

--- Source node 1 (score=0.792) ---
Metadata: {'page_label': '1', 'file_name': 'project_overview.pdf', 'file_path': '/content/demo_pdfs/project_overview.pdf', 'file_type': 'application/pdf', 'file_size': 1830, 'creation_date': '2026-06-04', 'last_modified_date': '2026-06-04'}
Project Overview: RAG Demo
This PDF describes example projects that combine time series forecasting, optimization, and agentic AI.
It is generate

In [24]:
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

In [25]:
faithfulness_eval = FaithfulnessEvaluator(llm=llm)
relevancy_eval = RelevancyEvaluator(llm=llm)

In [26]:
q = "Summarize the key ideas about agentic AI and RAG mentioned in these PDFs."
resp = query_engine.query(q)

faithfulness_result = faithfulness_eval.evaluate_response(
    query=q,
    response=resp, # Pass the full response object here
)

relevancy_result = relevancy_eval.evaluate_response(
    query=q,
    response=resp, # Pass the full response object here
)

print("Faithfulness score:", faithfulness_result.score)
print("Faithfulness feedback:", faithfulness_result.feedback)
print("\nRelevancy score:", relevancy_result.score)
print("Relevancy feedback:", relevancy_result.feedback)

Faithfulness score: 1.0
Faithfulness feedback: YES

Relevancy score: 1.0
Relevancy feedback: YES
